# Sesión 4 · Práctica guiada: una tabla de vivienda por provincia y vuestro primer módulo

Esta práctica la hacemos juntos, paso a paso: el profesor escribe y explica cada celda, y tú la reproduces en tu notebook.

Objetivo: construir una tabla anual por provincia con hipotecas por mil habitantes, importe medio y esfuerzo respecto a la renta, y terminar guardando vuestras funciones en un fichero propio.

In [ ]:
# Preparacion: localiza la carpeta de la asignatura (y la descarga si estas en Google Colab).
# Ejecuta esta celda primero; no hace falta entenderla todavia.
import os, subprocess
from pathlib import Path

REPO = "https://github.com/luisaugustos/ppy-analisis-social.git"


def _buscar_carpeta():
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        for candidata in (carpeta, carpeta / "ppy-analisis-social"):
            if (candidata / "datos" / "ORIGEN.md").is_file():
                return candidata
    return None


raiz = _buscar_carpeta()
if raiz is None:
    subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
    raiz = Path("ppy-analisis-social").resolve()

os.chdir(raiz)
print("Carpeta de trabajo lista. Los datos estan en la subcarpeta 'datos'.")

In [ ]:
from pathlib import Path

import pandas as pd

hipotecas = pd.read_csv("datos/hipotecas.csv", dtype={"cod_prov": str}, parse_dates=["fecha"])
poblacion = pd.read_csv("datos/poblacion.csv", dtype={"cod_prov": str})
renta = pd.read_csv("datos/renta.csv", dtype={"cod_prov": str})

print(hipotecas.shape, poblacion.shape, renta.shape)

## Ejercicio 1 · Resumen anual

A partir de `hipotecas` (sin el total nacional, `cod_prov` distinto de `"00"`), cread `anual` con una fila por provincia y año y las columnas `hipotecas`, `importe_miles` y `meses`. Quedaos con los años de 12 meses.

In [ ]:
# TODO 1
...

assert anual.groupby("anyo")["cod_prov"].nunique().eq(52).all()
anual.head(3)

## Ejercicio 2 · Unir la población

1. Unid `anual` con `poblacion` por `cod_prov` y `anyo`, con `how="left"`, `validate="one_to_one"` e `indicator=True`.
2. Mostrad en qué años no hay dato de población.
3. Rellenad esos años con la última población conocida de cada provincia y eliminad la columna `_merge`.

In [ ]:
# TODO 2
...

assert len(tabla) == len(anual) and tabla["poblacion"].notna().all()

## Ejercicio 3 · Dos indicadores

Añadid a `tabla`:

- `tasa_mil`: hipotecas por cada mil habitantes;
- `importe_medio`: euros por hipoteca (ojo: `importe_miles` está en miles de euros).

In [ ]:
# TODO 3
...

assert tabla["tasa_mil"].between(1, 40).all() and tabla["importe_medio"].min() > 10_000
tabla[tabla["provincia"].isin(["Salamanca", "Granada"])].tail(4).round(2)

## Ejercicio 4 · Unir la renta

Unid la renta neta media por persona. La tabla de renta no cubre todos los años: usad `how="inner"` y comprobad cuántas filas se pierden. Después calculad `esfuerzo`: importe medio dividido por la renta (cuántos años de renta de una persona equivale la hipoteca media).

In [ ]:
# TODO 4
...

assert len(completa) < antes and completa["esfuerzo"].notna().all()
completa.nlargest(5, "esfuerzo")[["provincia", "anyo", "importe_medio", "renta_neta_persona", "esfuerzo"]].round(2)

## Ejercicio 5 · Por comunidad autónoma

Calculad la `tasa_mil` de cada comunidad autónoma y año. Cuidado: **no** es la media de las tasas de sus provincias; hay que sumar hipotecas y población y dividir después. Mostrad el resultado en formato ancho, con los años en columnas.

In [ ]:
# TODO 5
...

ancho.iloc[:, -4:].sort_values(ancho.columns[-1], ascending=False).head(6)

## Ejercicio 6 · Vuestro módulo

Guardad en un fichero `mi_vivienda.py` dos funciones:

- `tabla_anual()`: hace lo de los ejercicios 1 a 3 y devuelve la tabla;
- `posicion(tabla, provincia, anyo)`: devuelve el puesto que ocupa una provincia ese año según `tasa_mil` (1 = la más alta).

La celda ya tiene el esqueleto: completad `posicion`.

In [ ]:
%%writefile mi_vivienda.py
"""Mis funciones para el caso de estudio de vivienda."""
import pandas as pd


def tabla_anual():
    """Hipotecas anuales por provincia con población, tasa por mil e importe medio."""
    hipotecas = pd.read_csv("datos/hipotecas.csv", dtype={"cod_prov": str}, parse_dates=["fecha"])
    poblacion = pd.read_csv("datos/poblacion.csv", dtype={"cod_prov": str})

    provincias = hipotecas[hipotecas["cod_prov"] != "00"].copy()
    provincias["anyo"] = provincias["fecha"].dt.year
    anual = (
        provincias
        .groupby(["cod_prov", "provincia", "ccaa", "anyo"], as_index=False)
        .agg(hipotecas=("hipotecas", "sum"), importe_miles=("importe_miles", "sum"), meses=("fecha", "count"))
    )
    anual = anual[anual["meses"] == 12].drop(columns="meses")

    tabla = anual.merge(poblacion[["cod_prov", "anyo", "poblacion"]], on=["cod_prov", "anyo"],
                        how="left", validate="one_to_one").sort_values(["cod_prov", "anyo"])
    tabla["poblacion"] = tabla.groupby("cod_prov")["poblacion"].ffill()
    tabla["tasa_mil"] = tabla["hipotecas"] / tabla["poblacion"] * 1000
    tabla["importe_medio"] = tabla["importe_miles"] / tabla["hipotecas"] * 1000
    return tabla


def posicion(tabla, provincia, anyo):
    """Puesto de una provincia en un año según su tasa de hipotecas por mil habitantes."""
    # TODO 6: filtrad el año, ordenad de mayor a menor tasa y buscad la posición de la provincia
    ...

In [ ]:
import importlib

import mi_vivienda

importlib.reload(mi_vivienda)            # por si habéis cambiado el fichero

mi_tabla = mi_vivienda.tabla_anual()
for nombre in ["Salamanca", "Granada"]:
    print(nombre, "ocupa el puesto", mi_vivienda.posicion(mi_tabla, nombre, 2025), "de 52")

assert 1 <= mi_vivienda.posicion(mi_tabla, "Salamanca", 2025) <= 52

## Ejercicio 7 · Guardar

Guardad `completa` en `resultados/vivienda_provincia_anual.parquet`, volved a leerla y comprobad que `cod_prov` sigue siendo texto de dos caracteres.

In [ ]:
# TODO 7
...

assert releida["cod_prov"].str.len().eq(2).all() and len(releida) == len(completa)

## Para cerrar

- ¿Qué decisiones discutibles habéis tomado al unir las fuentes? ¿Están escritas en el *docstring* de vuestras funciones?
- **Entregable 1 (sesiones 2–4):** se publica hoy. Consultad `entregables/entregable_1.md`.